# 09. SacreBLEU, chrF++, & COMET Benchmark Evaluation

**Requires GPU.** This notebook downloads an 8B-parameter model. Both **Qwen/Qwen2.5-7B-Instruct** and **mistralai/Mistral-7B-Instruct-v0.3** are fully open on HuggingFace Hub (no authentication needed).

Run on Colab with a GPU runtime -- see the setup cell below, which auto-clones the repo when a Colab GPU is detected.

Evaluates one model/checkpoint against the fixed `master_test.csv` split and saves results for the ablation study (notebook 11).

In [ ]:
# ============================================================
# PATH & ENVIRONMENT BOOSTER — Guarantees project path setup
# ============================================================
import os, sys, site, urllib.request, zipfile, glob

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
for conda_site in glob.glob('/opt/conda/lib/python3.*/site-packages'):
    if conda_site not in sys.path:
        sys.path.insert(0, conda_site)

try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

home     = os.path.expanduser('~')
proj_dir = os.path.join(home, 'Ekegusii-LLM-Translation-main')
tag_file = os.path.join(proj_dir, 'configs', 'models', 'v5_ready.tag')

# Download ONLY if tag_file is missing (prevents file modification during active runs)
if not os.path.isfile(tag_file):
    try:
        print('🔄 Syncing code from GitHub main branch...')
        zip_path = os.path.join(home, 'repo.zip')
        urllib.request.urlretrieve('https://github.com/aykahsay/Ekegusii-LLM-Translation/archive/refs/heads/main.zip', zip_path)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(home)
        os.remove(zip_path)
        print('✅ Code synced to latest version!')
    except Exception as exc:
        print(f'⚠️ Notice: {exc} (using local files)')
else:
    print('✅ Codebase up to date.')

if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')


In [ ]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


In [ ]:
MODEL_NAME = 'qwen'              # 'qwen' or 'mistral'
EXPERIMENT_ID = 'E4_Trilingual'  # must match a completed training run, or None for zero-shot (E0)
ADAPTER_PATH = f'checkpoints/{MODEL_NAME}/{EXPERIMENT_ID}/best' if EXPERIMENT_ID else None

In [ ]:
from src.cli.evaluate import run_evaluate

results = run_evaluate(MODEL_NAME, source_lang='English', target_lang='Ekegusii', adapter_path=ADAPTER_PATH)
results

## Save results for the ablation study

In [ ]:
import json
from pathlib import Path

out_dir = Path('experiments') / (EXPERIMENT_ID or 'E0_Baseline')
out_dir.mkdir(parents=True, exist_ok=True)
results_path = out_dir / 'results.json'

existing = json.loads(results_path.read_text()) if results_path.exists() else {'experiment_id': EXPERIMENT_ID}
existing[MODEL_NAME] = results
results_path.write_text(json.dumps(existing, indent=2))
print(f'Saved to {results_path}')

## COMET (optional, slower -- downloads a ~1.7GB checkpoint on first use)

In [ ]:
from src.experiments.base import BaseExperiment
from src.master_corpus.manager import MasterCorpusManager
from src.evaluation.comet import CometEvaluator
from src.models.qwen.inference import translate_with_qwen
from src.models.mistral.inference import translate_with_mistral

class _EvalHelper(BaseExperiment):
    experiment_id = 'notebook09'
    def build_training_tasks(self): raise NotImplementedError

helper = _EvalHelper(MasterCorpusManager())
test_pairs = helper.build_test_pairs('English', 'Ekegusii')
sources, references = test_pairs['source'].tolist(), test_pairs['target'].tolist()

translate_fn = translate_with_qwen if MODEL_NAME == 'qwen' else translate_with_mistral
predictions = translate_fn(sources, 'English', 'Ekegusii', adapter_path=ADAPTER_PATH)

comet_result = CometEvaluator().compute(predictions, references, sources)
print(f"Mean COMET: {comet_result['mean_score']:.4f}")